# Módulo 5 — Modelamiento de series temporales

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> AR, MA, ARMA, ARIMA, SARIMA; selección con AIC/BIC; análisis de residuos y Ljung-Box.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # gráficos estadísticos (preinstalado en Colab)

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de la serie preparada

Partimos de la versión ya limpia y ordenada (`_LIMPIO.csv`). En un flujo real usarías la salida del Módulo 1.

In [ ]:
df = cargar_datos('datos_proceso_planta_LIMPIO.csv', parse_dates=['Fecha'])
df = df.sort_values('Fecha').set_index('Fecha')
df = df.asfreq('h')   # eje horario regular; expone huecos como NaN
df.head()

In [ ]:
serie = df['Recuperacion_pct'].dropna()

## 2. Candidatos a partir de ACF/PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(serie.diff().dropna(), lags=40, ax=ax[0])
plot_pacf(serie.diff().dropna(), lags=40, ax=ax[1], method='ywm')
plt.show()

## 3. Ajuste y comparación AIC/BIC

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

ordenes = [(1,1,0), (2,1,0), (1,1,1), (2,1,1), (3,1,1)]
filas = []
ajustes = {}
for orden in ordenes:
    r = ARIMA(serie, order=orden).fit()
    ajustes[orden] = r
    filas.append({'orden': str(orden), 'AIC': r.aic, 'BIC': r.bic})
tabla = pd.DataFrame(filas).sort_values('AIC').reset_index(drop=True)
tabla

## 4. Residuos del mejor candidato

In [ ]:
mejor = tabla.loc[0, 'orden']
res = ajustes[eval(mejor)]
print(res.summary())

In [ ]:
res.plot_diagnostics(figsize=(11, 8)); plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
acorr_ljungbox(res.resid.dropna(), lags=[12, 24, 48], return_df=True)

## 5. Componente estacional: SARIMA

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
sar = SARIMAX(serie, order=(1,1,1), seasonal_order=(1,0,1,24),
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print('AIC SARIMA:', round(sar.aic, 1))
acorr_ljungbox(sar.resid.iloc[50:], lags=[24, 48], return_df=True)

## 6. (Opcional) búsqueda automática

In [ ]:
%pip -q install pmdarima
import pmdarima as pm
auto = pm.auto_arima(serie, seasonal=True, m=24, d=1,
                     stepwise=True, suppress_warnings=True, trace=True)
auto.summary()

## 7. Hoja de selección (complétala)

| Modelo | AIC | BIC | Ljung-Box (p) | ¿pasa a M6? |
|---|---|---|---|---|
| ARIMA(_ , _ , _) | | | | |
| ARIMA(_ , _ , _) | | | | |
| SARIMA(_)(_ )24 | | | | |

## Actividades sugeridas

1. Repite la comparación de órdenes para `Tonelaje_tph` (usa d y transformación de M4).
2. ¿El SARIMA reduce la autocorrelación residual respecto del ARIMA no estacional?
3. Interpreta el coeficiente AR estimado en términos de persistencia, no de causalidad.
4. Elige 2 modelos razonables y justifícalos; NO elijas solo por AIC mínimo.

---
## Cierre

Tenemos 1–2 modelos estadísticamente razonables. El Módulo 6 hace la prueba que importa: predecir datos nunca vistos.